## Grafos y comunidades: Funciones

Constantes y funciones compartidas

### Imports

In [ ]:
import os, time
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

In [ ]:
import json
import subprocess, sys
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components, breadth_first_order
from scipy.sparse.linalg import eigsh
from scipy.cluster.vq import kmeans2
from scipy.stats import rankdata
from pyspark.sql import Window
try:
    import networkx as nx
    from networkx.algorithms.community import louvain_communities
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "networkx>=3.1"])
    import networkx as nx
    from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community import girvan_newman

### Constantes

- `datos_dir`: datos que pasan de un notebook al siguiente

- `final_dir`: tablas finales de `01_Trans` (`participantes`, `procesos`, `hechos`)

- `clus_dir`: salidas de `02_Clustering` (tipologías y etiquetas de KMeans)

- `pr_dir`: datos de `04_PageRank` (PageRank y HITS por actor)

- `GF_PKG`: GraphFrames para Spark 3.5 / Scala 2.12, la versión de la imagen del cluster

In [ ]:
SEED = 42
datos_dir = Path.cwd() / "temp" / "datos"
datos_dir.mkdir(parents=True, exist_ok=True)
final_dir = artef_dir.parent / "final"
clus_dir = proyecto / "02_Clustering" / "temp"
pr_dir = proyecto / "04_PageRank" / "temp" / "datos"
GF_PKG = "graphframes:graphframes:0.8.3-spark3.5-s_2.12"
GF_REPO = "https://repos.spark-packages.org"

### Parámetros elegidos

| Parámetro | Valor | Por qué | Sección |
|---|---|---|---|
| Grafo | co-participación proveedor–proveedor, **no dirigido** | Dos proveedores se unen si compiten en ≥ 2 procesos de 2 a 30 postores; una comunidad es un grupo que se encuentra una y otra vez | `a_grafo` |
| `PESO` | `peso` = Σ 1/(n − 1) | Coincidir en un proceso de 30 postores pesa menos que en uno de 3 | `a_grafo` |
| `MIN_COP`, `TOPE_GRADO` | `config.json` | Mínimo de procesos compartidos y percentil de grado desde el que un proveedor es hub; definen el núcleo | `a_grafo` |
| `N_MIN`, `N_MAX`, `E_MAX` | 1,500 / 8,000 / 200,000 | Núcleo representativo que cabe en los algoritmos locales | `a_grafo` |
| `GAMMA` | `config.json` | Resolución de Louvain más estable entre semillas cerca del máximo de Q | `b_particion` |
| `K_ESPECTRAL` | `config.json` | k con mayor Q (se contrasta con el eigengap) | `b_particion` |
| `K_BIGCLAM`, `DELTA` | `config.json` | K por AUC en aristas retenidas; δ teórico de Yang & Leskovec (√(−log(1 − ε)), ε = densidad) | `c_solapadas` |
| `ITER_LPA`, `TOL_LPA` | 20 / 0.001 | LPA en Spark: corta cuando cambia < 0.1% de los nodos | `d_escalabilidad` |
| `MIN_TAM` | 10 | Tamaño mínimo para perfilar una comunidad | `b_particion`, `e_analisis` |
| `REPS` | 100 | Permutaciones del NMI nulo | `e_analisis` |
| `TOP_K` | 20 | Comunidades en tablas y mapas de calor | todos |
| `P_ALTO` | 0.99 | El mismo de `04_PageRank` para "muy central" | `e_analisis` |
| `UMBRAL_HHI`, `MIN_PROC_ENT` | 0.25 / 5 | Los mismos de `02_Clustering` | `e_analisis` |

In [ ]:
PESO = "peso"
N_MIN, N_MAX, E_MAX = 1_500, 8_000, 200_000
ITER_LPA, TOL_LPA = 20, 0.001
MIN_TAM = 10
REPS = 100
TOP_K = 20
P_ALTO = 0.99
UMBRAL_HHI = 0.25
MIN_PROC_ENT = 5

### Configuración entre notebooks

`config.json` guarda lo que cada notebook decide (núcleo, peso, γ, k, K, δ…)

In [ ]:
def leer_config():
    ruta = datos_dir / "config.json"
    return json.loads(ruta.read_text()) if ruta.exists() else {}

def guardar_config(config):
    (datos_dir / "config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False))

### Spark con GraphFrames

- Se pasa `--packages` por `PYSPARK_SUBMIT_ARGS`, igual que `pyspark --packages`, sin tocar `crear_spark`

- Si no hay red para bajar el paquete, se sigue sin GraphFrames: la implementación propia no depende de él

In [ ]:
def crear_spark_grafos(nombre):
    os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {GF_PKG} --repositories {GF_REPO} pyspark-shell"
    try:
        s = crear_spark(nombre)
    except Exception as e:
        print(f"Sin GraphFrames ({type(e).__name__}): se continúa con la implementación propia")
        os.environ.pop("PYSPARK_SUBMIT_ARGS", None)
        return crear_spark(nombre), False
    try:
        from graphframes import GraphFrame
    except ImportError:
        for jar in s.sparkContext.getConf().get("spark.jars", "").split(","):
            if "graphframes" in jar:
                s.sparkContext.addPyFile(jar)
        try:
            from graphframes import GraphFrame
        except ImportError:
            return s, False
    return s, True

### Grafo

- `pesos`: `binario`, `n_coparticipa` o `peso`

- `matriz`: adyacencia simétrica N × N a partir de aristas (i, j) guardadas una vez

- `gigante`: máscara de la componente conexa más grande

- `bola`: los `tam` primeros nodos de un BFS desde una semilla (por defecto, el de mayor grado); da subgrafos conexos para muestras y tiempos

- `a_networkx`: la misma matriz como grafo de networkx (atributo `weight`)

In [ ]:
def pesos(a, peso):
    if peso == "binario":
        return np.ones(len(a))
    return a[peso].astype(float).to_numpy()

def matriz(a, n, peso):
    W = sp.coo_matrix((pesos(a, peso), (a.i, a.j)), shape=(n, n))
    return (W + W.T).tocsr()

def gigante(A):
    _, comp = connected_components(A, directed=False)
    return comp == np.bincount(comp).argmax()

def bola(A, tam, semilla=None):
    if semilla is None:
        semilla = int(np.asarray((A > 0).sum(axis=1)).ravel().argmax())
    orden = breadth_first_order(A, semilla, directed=False, return_predecessors=False)[:tam]
    return np.sort(orden)

def a_networkx(A):
    return nx.from_scipy_sparse_array(A)

### Calidad de una partición

- `modularidad`: Q = Σ_c [ e_c / 2m − γ (vol_c / 2m)² ], con e_c el peso interno (contado dos veces) y vol_c la suma de fuerzas

- `por_comunidad`: tamaño, aristas, peso interno, conductancia (corte / min(vol_c, 2m − vol_c)) y densidad de cada comunidad

- `resumir`: una fila por partición para las tablas de comparación

- `conductancia_conjunto`: lo mismo para un conjunto de nodos (comunidades solapadas)

In [ ]:
def compactar(lab):
    return np.unique(np.asarray(lab), return_inverse=True)[1]

def indicadora(lab):
    lab = compactar(lab)
    return sp.csr_matrix((np.ones(len(lab)), (np.arange(len(lab)), lab)))

def modularidad(A, lab, gamma=1.0):
    H = indicadora(lab)
    dos_m = A.sum()
    interno = (H.T @ A @ H).diagonal()
    vol = H.T @ np.asarray(A.sum(axis=1)).ravel()
    return float(interno.sum() / dos_m - gamma * ((vol / dos_m) ** 2).sum())

def por_comunidad(A, lab):
    H = indicadora(lab)
    Ab = (A > 0).astype(float)
    tam = np.asarray(H.sum(axis=0)).ravel()
    interno = (H.T @ A @ H).diagonal()
    vol = H.T @ np.asarray(A.sum(axis=1)).ravel()
    aristas = (H.T @ Ab @ H).diagonal() / 2
    den = np.minimum(vol, A.sum() - vol)
    cond = np.divide(vol - interno, den, out=np.zeros(len(vol)), where=den > 0)
    dens = np.divide(aristas, tam * (tam - 1) / 2, out=np.zeros(len(tam)), where=tam > 1)
    return pd.DataFrame({"comunidad": np.arange(len(tam)), "tamano": tam.astype(int), "aristas": aristas.astype(int),
                         "peso_interno": interno / 2, "conductancia": cond, "densidad": dens})

def resumir(A, lab, metodo, seg=np.nan, gamma=1.0):
    c = por_comunidad(A, lab)
    grandes = c[c.tamano >= MIN_TAM]
    return {"metodo": metodo, "comunidades": len(c), "comunidades_grandes": len(grandes),
            "pct_nodos_en_mayor": 100 * c.tamano.max() / c.tamano.sum(),
            "pct_nodos_en_grandes": 100 * grandes.tamano.sum() / c.tamano.sum(),
            "Q": modularidad(A, lab, gamma), "cobertura": c.peso_interno.sum() / (A.sum() / 2),
            "conductancia_med": grandes.conductancia.median(), "seg": seg}

def conductancia_conjunto(A, nodos, d=None):
    d = np.asarray(A.sum(axis=1)).ravel() if d is None else d
    vol, interno = d[nodos].sum(), A[nodos][:, nodos].sum()
    den = min(vol, d.sum() - vol)
    return float((vol - interno) / den) if den > 0 else 0.0

### Comparación de particiones

- `nmi`: información mutua normalizada por la media de las entropías (0 = independientes, 1 = iguales)

- `ari`: índice de Rand ajustado (0 = azar)

- `nmi_nulo`: media y p95 del NMI cuando se permuta el atributo; un NMI por encima del p95 no es casualidad

- `pureza`: fracción de nodos cuyo atributo coincide con el más frecuente de su comunidad

In [ ]:
def _contingencia(x, y):
    _, xi = np.unique(x, return_inverse=True)
    _, yi = np.unique(y, return_inverse=True)
    return sp.coo_matrix((np.ones(len(xi)), (xi, yi))).tocsr()

def _entropia(c):
    p = c[c > 0] / c.sum()
    return float(-(p * np.log(p)).sum())

def nmi(x, y):
    M = _contingencia(x, y); n = M.sum()
    fa, fb = np.asarray(M.sum(1)).ravel(), np.asarray(M.sum(0)).ravel()
    M = M.tocoo()
    mi = float((M.data / n * np.log(M.data * n / (fa[M.row] * fb[M.col]))).sum())
    hx, hy = _entropia(fa), _entropia(fb)
    return 2 * mi / (hx + hy) if hx + hy > 0 else 1.0

def ari(x, y):
    M = _contingencia(x, y); n = M.sum()
    c2 = lambda z: float((z * (z - 1) / 2).sum())
    sij, sa, sb = c2(M.data), c2(np.asarray(M.sum(1)).ravel()), c2(np.asarray(M.sum(0)).ravel())
    esp = sa * sb / (n * (n - 1) / 2)
    return (sij - esp) / ((sa + sb) / 2 - esp) if (sa + sb) / 2 != esp else 1.0

def nmi_nulo(x, y, reps=REPS, seed=SEED):
    rng = np.random.default_rng(seed); y = np.asarray(y)
    nulo = np.array([nmi(x, rng.permutation(y)) for _ in range(reps)])
    return float(nulo.mean()), float(np.quantile(nulo, .95))

def pureza(lab, attr):
    t = pd.DataFrame({"c": lab, "a": attr}).dropna()
    return float(t.groupby(["c", "a"]).size().groupby(level=0).max().sum() / len(t))

def pct_si(s):
    return 100 * pd.to_numeric(pd.Series(s), errors="coerce").astype(float).mean()

### Algoritmos (local)

- `louvain`: networkx (`louvain_communities`) con peso y resolución γ

- `espectro`: los k mayores autovalores de D^(−1/2) A D^(−1/2) (equivale a los k menores del laplaciano normalizado, λ_L = 1 − λ)

- `particion_espectral`: filas del embedding normalizadas + KMeans (Ng–Jordan–Weiss)

- `girvan_newman_q`: niveles de Girvan–Newman con su Q, hasta `max_com` comunidades

- `lpa_local`: Label Propagation síncrona con pesos e inercia (el nodo vota por su etiqueta con el peso medio de sus aristas); es la referencia del LPA en Spark

In [ ]:
def de_conjuntos(comunidades, n):
    lab = np.full(n, -1)
    for c, nodos in enumerate(sorted(comunidades, key=len, reverse=True)):
        lab[list(nodos)] = c
    return lab

def louvain(G, gamma=1.0, seed=SEED):
    return de_conjuntos(louvain_communities(G, weight="weight", resolution=gamma, seed=seed), G.number_of_nodes())

def espectro(A, k):
    d = np.asarray(A.sum(axis=1)).ravel()
    Dm = sp.diags(1 / np.sqrt(d))
    val, vec = eigsh(Dm @ A @ Dm, k=k, which="LA")
    o = np.argsort(-val)
    return 1 - val[o], vec[:, o]

def particion_espectral(vec, k, seed=SEED):
    X = vec[:, :k] / np.maximum(np.linalg.norm(vec[:, :k], axis=1, keepdims=True), 1e-12)
    _, lab = kmeans2(X, k, minit="++", seed=seed)
    return compactar(lab)

def girvan_newman_q(G, A, max_com, max_seg=900):
    niveles, t0 = [], time.time()
    for comunidades in girvan_newman(G):
        lab = de_conjuntos(comunidades, G.number_of_nodes())
        niveles.append({"comunidades": len(comunidades), "Q": modularidad(A, lab), "seg": time.time() - t0, "lab": lab})
        if len(comunidades) >= max_com or time.time() - t0 > max_seg:
            break
    return niveles

def lpa_local(i, j, w, n, iters=ITER_LPA, tol=TOL_LPA, inercia=True):
    ii, jj, ww = np.r_[i, j], np.r_[j, i], np.r_[w, w].astype(float)
    grado = np.bincount(ii, minlength=n)
    activos = np.flatnonzero(grado > 0)
    propios = activos if inercia else np.zeros(0, int)
    voto = (np.bincount(ii, ww, n) / np.maximum(grado, 1))[propios]
    lab = np.arange(n)
    for k in range(1, iters + 1):
        d = pd.DataFrame({"id": np.r_[jj, propios], "lab": np.r_[lab[ii], lab[propios]], "v": np.r_[ww, voto]})
        d = d.groupby(["id", "lab"], sort=False).v.sum().round(9).reset_index()
        d = d.sort_values(["id", "v", "lab"], ascending=[True, False, True]).drop_duplicates("id")
        nuevo = lab.copy()
        nuevo[d.id.to_numpy()] = d.lab.to_numpy()
        cambio = float((nuevo[activos] != lab[activos]).mean())
        lab = nuevo
        if cambio < tol:
            break
    return lab, k, cambio

### BigCLAM

- `bigclam`: ascenso de gradiente proyectado (F ≥ 0) sobre la log-verosimilitud
  - L(F) = Σ_(u,v)∈E log(1 − exp(−F_u·F_v)) − Σ_(u,v)∉E F_u·F_v
  - ∇_u = Σ_v∈N(u) F_v / (1 − exp(−F_u·F_v)) − Σ_v F_v + F_u
  - El paso se reduce a la mitad si L baja y crece 1.5× si sube

- `semillas_bigclam`: inicializa cada comunidad con un vecindario (nodo + vecinos) de baja conductancia, como Yang & Leskovec (2013)

- `delta_bigclam`: δ = √(−log(1 − ε)), con ε = densidad del grafo

- `separar_aristas`, `puntaje`, `auc`: aristas retenidas y AUC (Mann–Whitney) para elegir K

In [ ]:
def bigclam(A, K, iters=500, tol=1e-6, F0=None, min_iter=50, paciencia=5, eta0=1e-3, seed=SEED):
    """BigCLAM (Yang & Leskovec 2013), grafo binario no dirigido.
    L(F) = Σ_{(u,v)∈E} log(1 − e^{−F_u·F_v}) − Σ_{(u,v)∉E} F_u·F_v"""
    A = sp.csr_matrix(A, dtype=float)
    A.eliminate_zeros()
    A.data[:] = 1.0
    n = A.shape[0]
    r, c = sp.triu(A, k=1).nonzero()
    F = (np.random.default_rng(seed).uniform(0, 0.1, (n, K)) if F0 is None
         else np.maximum(np.asarray(F0, dtype=float), 0.0))

    def pares(F):
        return np.maximum(np.einsum("ij,ij->i", F[r], F[c]), 1e-10)

    def verosimilitud(F):
        d = pares(F); s = F.sum(0)
        no_aristas = (s @ s - np.einsum("ij,ij->", F, F)) / 2 - d.sum()
        return np.log(-np.expm1(-d)).sum() - no_aristas

    def gradiente(F):
        w = 1.0 / np.expm1(pares(F)) + 1.0            # ∂/∂F_u = Σ_v∈N(u) F_v (c_uv + 1) − ΣF + F_u
        W = sp.coo_matrix((np.r_[w, w], (np.r_[r, c], np.r_[c, r])), shape=(n, n)).tocsr()
        return W @ F - F.sum(0) + F

    L = verosimilitud(F); hist = [L]; eta = eta0; quietos = 0
    for it in range(1, iters + 1):
        g = gradiente(F)
        while True:
            F_n = np.maximum(F + eta * g, 0.0)
            L_n = verosimilitud(F_n)
            if L_n > L or eta < 1e-14:
                break
            eta /= 2
        if L_n <= L:                                  # ni con el paso mínimo mejora: óptimo local
            break
        mejora = (L_n - L) / abs(L)
        F, L = F_n, L_n; hist.append(L); eta *= 1.5
        quietos = quietos + 1 if mejora < tol else 0
        if it >= min_iter and quietos >= paciencia:
            break
    return F, np.array(hist)

def semillas_bigclam(A, K, bloque=500):
    Ab = sp.csr_matrix((A > 0).astype(float))
    n = Ab.shape[0]
    d = np.asarray(Ab.sum(1)).ravel()
    tri = np.concatenate([np.asarray((Ab[s:s + bloque] @ Ab).multiply(Ab[s:s + bloque]).sum(1)).ravel()
                          for s in range(0, n, bloque)]) / 2
    vol = d + Ab @ d
    cond = (vol - 2 * (d + tri)) / np.maximum(np.minimum(vol, d.sum() - vol), 1)
    F, usados, k = np.zeros((n, K)), np.zeros(n, bool), 0
    for u in np.argsort(cond):
        if usados[u]:
            continue
        vec = Ab[u].indices
        F[u, k] = 1; F[vec, k] = 1
        usados[u] = True; usados[vec] = True
        k += 1
        if k == K:
            break
    return F

def delta_bigclam(A):
    n = A.shape[0]
    eps = (A > 0).sum() / (n * (n - 1))
    return float(np.sqrt(-np.log(1 - eps)))

def separar_aristas(A, frac=0.1, seed=SEED):
    A = sp.csr_matrix(A); n = A.shape[0]
    r, c = sp.triu(A, k=1).nonzero()
    rng = np.random.default_rng(seed)
    test = rng.random(len(r)) < frac
    T = sp.coo_matrix((np.ones((~test).sum()), (r[~test], c[~test])), shape=(n, n))
    A_tr = (T + T.T).tocsr()
    existentes, neg = set(zip(r.tolist(), c.tolist())), set()
    while len(neg) < test.sum():
        i, j = sorted(rng.integers(0, n, 2).tolist())
        if i != j and (i, j) not in existentes:
            neg.add((i, j))
    return A_tr, np.c_[r[test], c[test]], np.array(sorted(neg))

def puntaje(F, pares):
    return (F[pares[:, 0]] * F[pares[:, 1]]).sum(1)

def auc(pos, neg):
    r = rankdata(np.r_[pos, neg])
    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))

### Otras partes

- `tipologia`: etiqueta KMeans de cada actor (de `02_Clustering/temp/datos`) con el nombre de su tipología (de `e_analisis`)

- `centralidad_04`: PageRank y HITS de cada proveedor en el bipartito de `04_PageRank`, con su percentil

In [ ]:
def tipologia(s, key):
    ruta_tip = clus_dir / "e_analisis" / "tablas" / "tipologias"
    if not ruta_tip.exists():
        raise FileNotFoundError("Falta 02_Clustering/temp/e_analisis/tablas/tipologias: correr e_analisis")
    nombres = pd.read_parquet(ruta_tip)
    nombres = nombres.loc[nombres.sujeto == s, ["kmeans", "tipologia"]]
    for f in (f"kmeans_{s}", f"prep_{s}"):
        p = clus_dir / "datos" / f
        if not p.exists():
            continue
        d = pd.read_parquet(p)
        if "kmeans" not in d.columns and "cluster" in d.columns:
            d = d.rename(columns={"cluster": "kmeans"})
        if "kmeans" in d.columns and key in d.columns:
            return d[[key, "kmeans"]].merge(nombres, on="kmeans", how="left")
    raise FileNotFoundError(f"Faltan las etiquetas KMeans de {s}: correr b_kmeans de 02_Clustering")

def centralidad_04():
    if not (pr_dir / "hits.parquet").exists():
        raise FileNotFoundError("Faltan los datos de 04_PageRank: correr a_grafo, b_pagerank y c_hits")
    c = (pd.read_parquet(pr_dir / "nodos.parquet")[["idx", "tipo", "clave"]]
           .merge(pd.read_parquet(pr_dir / "pagerank.parquet")[["idx", "pr"]], on="idx")
           .merge(pd.read_parquet(pr_dir / "hits.parquet"), on="idx"))
    c = c[c.tipo == "proveedor"].copy()
    c["pct_pr"], c["pct_hits"] = c.pr.rank(pct=True), c.hits.rank(pct=True)
    return c[["clave", "pr", "hits", "pct_pr", "pct_hits"]]

In [ ]:
def rellenar_vacias(t):
    t = t.copy()
    for c in t.columns:
        if t[c].dtype == object and t[c].isna().all():
            t[c] = "sin dato"
    return t


_guardar_tablas_base = guardar_tablas

def guardar_tablas(tablas, *args, **kwargs):
    vacias = [k for k, t in tablas.items() if len(t) == 0]
    if vacias:
        print(f"Aviso: tablas sin filas, no se escriben: {', '.join(vacias)}")
    return _guardar_tablas_base({k: rellenar_vacias(t) for k, t in tablas.items() if len(t) > 0}, *args, **kwargs)

In [ ]:
def muestra_comunidades(A, lab, n_obj=300, n_com=4, seed=SEED):
    """Muestra con estructura conocida: una comunidad semilla y sus n_com − 1 vecinas más unidas;
    de cada una, ~n_obj / n_com nodos por BFS interno desde el nodo más unido a las otras (conserva los puentes)."""
    lab = np.asarray(lab)
    H = indicadora(lab)
    W = (H.T @ A @ H).toarray(); np.fill_diagonal(W, 0)
    tam = np.bincount(lab)
    ok = np.flatnonzero(tam >= n_obj // n_com)
    semilla = int(np.random.default_rng(seed).choice(ok))
    vecinas = [int(c) for c in np.argsort(-W[semilla]) if c in ok and W[semilla, c] > 0][:n_com - 1]
    elegidas = [semilla] + vecinas
    muestra = []
    for c in elegidas:
        nodos_c = np.flatnonzero(lab == c)
        otras = np.flatnonzero(np.isin(lab, [x for x in elegidas if x != c]))
        hacia = np.asarray(A[nodos_c][:, otras].sum(axis=1)).ravel()
        sub = bola(A[nodos_c][:, nodos_c], n_obj // n_com, semilla=int(hacia.argmax()))
        muestra.append(nodos_c[sub])
    return np.sort(np.concatenate(muestra)), elegidas